# 步骤 1: 准备工作与环境设置

**目标:** 导入所有必要的库，加载spaCy模型，并定义数据文件路径。这是整个流程的起点。

In [5]:
# 导入必要的库
import pandas as pd
import spacy
import re
import time
from tqdm.auto import tqdm

# 初始化tqdm以便在pandas apply中显示进度条
tqdm.pandas()

# --- 配置区 ---
# 请将这里的路径修改为你的实际文件路径
# Parquet格式的输入文件
INPUT_FILE_PATH = "/Users/tong/WSJ_China/data_raw/first_100_rows.parquet"
# 处理后保存的中间文件
OUTPUT_FILE_PATH = "data_processed/01_data_processed/wsj_data_lemmatized.parquet"

print("--- 环境准备 ---")
print(f"输入文件路径: {INPUT_FILE_PATH}")
print(f"输出文件路径: {OUTPUT_FILE_PATH}")

# --- 加载spaCy模型 ---
# 加载中等大小的英文模型。'disable'参数可以关闭不需要的管道组件，极大地提高处理速度。
# 我们在这里只需要分词器(tokenizer)和词形还原器(lemmatizer)。
print("\n正在加载spaCy模型 'en_core_web_sm'...")
start_time = time.time()
# 增加一个错误处理，如果模型没安装会提示
try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    print(f"spaCy模型加载成功！耗时: {time.time() - start_time:.2f} 秒")
except OSError:
    print("错误: spaCy模型 'en_core_web_sm' 未安装。")
    print("请在你的终端或命令行中运行: python -m spacy download en_core_web_sm")
    # 如果在notebook中，可以取消下面一行的注释来安装
    # !python -m spacy download en_core_web_md
    # nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

--- 环境准备 ---
输入文件路径: /Users/tong/WSJ_China/data_raw/first_100_rows.parquet
输出文件路径: data_processed/01_data_processed/wsj_data_lemmatized.parquet

正在加载spaCy模型 'en_core_web_sm'...
spaCy模型加载成功！耗时: 0.38 秒


# 步骤 2: 加载并探查数据

**目标:** 从Parquet文件中读取数据，并进行初步的探查，了解数据结构和质量。

In [6]:
print(f"\n--- 步骤 2: 加载数据 ---")
print(f"正在从 {INPUT_FILE_PATH} 读取Parquet文件...")

start_time = time.time()
# 使用pandas读取Parquet文件
df = pd.read_parquet(INPUT_FILE_PATH)
load_time = time.time() - start_time
print(f"数据加载完成！耗时: {load_time:.2f} 秒。")

# 探查数据
print(f"\n数据概览:")
print(f"数据形状 (行, 列): {df.shape}")
print("\n前3行数据:")
# 设置pandas显示选项，防止文本被截断
pd.set_option('display.max_colwidth', 80)
print(df.head(3))
print("\n数据信息 (列名, 非空值, 类型):")
df.info(verbose=True, show_counts=True)

# 你的列名是 'DATE' 和 'CONTENT'，我们将其重命名为更通用的名字
df.rename(columns={'DATE': 'date', 'CONTENT': 'full_text'}, inplace=True)
print("\n已将列名'DATE', 'CONTENT'分别重命名为'date', 'full_text'")


--- 步骤 2: 加载数据 ---
正在从 /Users/tong/WSJ_China/data_raw/first_100_rows.parquet 读取Parquet文件...
数据加载完成！耗时: 0.30 秒。

数据概览:
数据形状 (行, 列): (100, 2)

前3行数据:
         DATE  \
0  1984-01-02   
1  1984-01-03   
2  1984-01-04   

                                                                           CONTENT  
0  Air Illinois Says It Hopes to Resume Some Service Monday\nCARBONDALE, Ill. -...  
1  A&P Says Settlement Over Its Pension Plan Is Affirmed by Court\nMONTVALE, N....  
2  A.H. Belo, Freedom Newspapers\nDALLAS -- A.H. Belo Corp. and Freedom Newspap...  

数据信息 (列名, 非空值, 类型):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   DATE     100 non-null    object
 1   CONTENT  100 non-null    object
dtypes: object(2)
memory usage: 1.7+ KB

已将列名'DATE', 'CONTENT'分别重命名为'date', 'full_text'


# 步骤 3: 轻量级文本清理

**目标:** 对`full_text`列进行基础的、计算成本较低的清理，包括转为小写和移除URL、邮件等明显噪声。

In [7]:
print(f"\n--- 步骤 3: 轻量级文本清理 ---")

# 定义一个函数来执行基础的文本清理
def basic_text_cleaning(text):
    # 确保输入是字符串
    if not isinstance(text, str):
        return ""

    # 1. 转为小写
    text = text.lower()

    # 2. 移除URL (http/https/www)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 3. 移除邮件地址
    text = re.sub(r'\S+@\S+', '', text)

    # 4. 移除包含数字的单词 (例如股票代码 'aa23', 或者日期 '1990s')
    # 这样可以避免后续词形还原处理它们
    text = re.sub(r'\b\w*\d\w*\b', '', text)

    # 5. 移除换行符和多余的空格
    text = re.sub(r'\s+', ' ', text).strip()

    return text

print("正在对 'full_text' 列进行基础清理...")
start_time = time.time()
# 将清理函数应用到 'full_text' 列
# 使用 .progress_apply 来显示进度条
df['cleaned_text'] = df['full_text'].progress_apply(basic_text_cleaning)
clean_time = time.time() - start_time
print(f"基础清理完成！耗时: {clean_time/60:.2f} 分钟。")

# 检查清理效果
print("\n清理效果对比 (第一篇文章):")
print("--- 原始文本 ---")
print(df['full_text'].iloc[0][:500] + "...")
print("\n--- 清理后文本 ---")
print(df['cleaned_text'].iloc[0][:500] + "...")


--- 步骤 3: 轻量级文本清理 ---
正在对 'full_text' 列进行基础清理...


100%|██████████| 100/100 [00:02<00:00, 42.42it/s]

基础清理完成！耗时: 0.04 分钟。

清理效果对比 (第一篇文章):
--- 原始文本 ---
Air Illinois Says It Hopes to Resume Some Service Monday
CARBONDALE, Ill. -- Air Illinois, the closely held airline that voluntarily ceased operations Dec. 15 because of government safety questions, hopes to resume limited operations by Monday, Roger Street, president, said. But he said it still isn't certain whether the airline will survive. The grounding followed an Oct. 11 crash that killed 10 people. If the Federal Aviation Administration recertifies the airline, it will resume jet service f...

--- 清理后文本 ---
air illinois says it hopes to resume some service monday carbondale, ill. -- air illinois, the closely held airline that voluntarily ceased operations dec. because of government safety questions, hopes to resume limited operations by monday, roger street, president, said. but he said it still isn't certain whether the airline will survive. the grounding followed an oct. crash that killed people. if the federal aviation administ

# 步骤 4: 应用spaCy进行词形还原 (Lemmatization)

**目标:** 对清理后的文本进行词形还原，将所有单词转换为它们的基本形态（词元），为后续的关键词匹配和建模做准备。这是整个流程中最耗时的一步。

In [ ]:
print(f"\n--- 步骤 4: 词形还原 (安全版本) ---")

# 定义一个使用spaCy进行词形还原的函数 (不移除停用词)
# nlp.pipe 提供了巨大的性能提升，因为它能批量处理文本
def lemmatize_text_pipe_safe(texts, batch_size=500):
    """
    使用 nlp.pipe 批量处理文本以进行词形还原。
    这个安全版本只做最基本的过滤，不移除停用词。
    """
    lemmatized_results = []
    # 使用tqdm包装nlp.pipe以获得总进度条
    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size), total=len(texts), desc="spaCy批量处理中"):
        # 提取每个token的词元。
        # 这里的过滤条件非常宽松：
        # 1. token.is_alpha: 确保是纯字母，过滤掉数字和标点。
        # 2. len(token.lemma_) > 1: 过滤掉单字符的词 (例如 's' 来自 's)
        # *** 注意：这里完全没有使用 token.is_stop ***
        lemmas = [token.lemma_ for token in doc if token.is_alpha and len(token.lemma_) > 1]

        # 将所有词元用空格连接成一个新的字符串
        lemmatized_results.append(" ".join(lemmas))

    return lemmatized_results

print("开始进行批量词形还原（安全模式，不移除停用词），这将花费较长时间，请耐心等待...")
start_time = time.time()

# 将整个'cleaned_text'列作为列表传入函数
lemmatized_texts = lemmatize_text_pipe_safe(df['cleaned_text'].tolist())

# 将结果添加回DataFrame
df['lemmatized_text'] = lemmatized_texts

lemmatize_time = time.time() - start_time
print(f"词形还原完成！总耗时: {lemmatize_time/60:.2f} 分钟 (或 {lemmatize_time/3600:.2f} 小时)。")

# 检查词形还原效果
print("\n词形还原效果对比 (第一篇文章):")
print("--- 清理后文本 ---")
print(df['cleaned_text'].iloc[0][:500] + "...")
print("\n--- 词形还原后文本 (安全版) ---")
# 打印一下，


--- 步骤 4: 词形还原 (安全版本) ---
开始进行批量词形还原（安全模式，不移除停用词），这将花费较长时间，请耐心等待...


spaCy批量处理中:   0%|          | 0/100 [00:00<?, ?it/s]

# 步骤 5: 保存处理后的中间结果

**目标:** 将包含`lemmatized_text`列的DataFrame保存为Parquet格式，以便在后续的流程中直接加载，无需重复上面耗时的步骤。

In [ ]:
print(f"\n--- 步骤 5: 保存中间结果 ---")
print(f"正在将处理后的数据保存到: {OUTPUT_FILE_PATH}")

try:
    start_time = time.time()
    columns_to_save = ['date', 'lemmatized_text']
    df[columns_to_save].to_parquet(OUTPUT_FILE_PATH, index=False)
    save_time = time.time() - start_time
    print(f"文件保存成功！耗时: {save_time:.2f} 秒。")
except Exception as e:
    print(f"错误: 文件保存失败。错误信息: {e}")